# Molecular Dynamics Simulation of Alanine Dipeptide

In this notebook, we will be using AMBER force fields.
The OpenMM documentation contains [a complete overview of included force fields](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#force-fields).
The corresponding data files can be viewed on the [GitHub repository](https://github.com/openmm/openmm/tree/master/wrappers/python/openmm/app/data).
These files are already included in every OpenMM installation, so you don't need to download them.

Just to avoid getting lost in the force field zoo, here are the key references to various AMBER force fields and some comments on the history of their development:

- 1997 `amber96.xml`: a slight modification of the backbone torsional parameters of AMBER ff94, based on *ab initio* calculations, not very significant. [REF](https://doi.org/10.1007/978-94-017-1120-3_2)

- 2003 `amber03.xml`: new atomic charges derived from *ab initio* calculations in a continuum solvent model, combined with refitted backbone torsional parameters, starting from AMBER ff99. In the literature referred to as AMBER ff03. [REF](https://doi.org/10.1002/jcc.10349)

- 2006 `amber99sb.xml`: a refitting of the AMBER ff99 backbone torsional parameters to *ab initio* data, to obtain a better balance between the stabilities of different secondary structure elements. In the literature referred to as AMBER ff99SB, where SB stands for Stony Brook. [REF](https://doi.org/10.1002/prot.21123)

- 2010 `amber99sbildn.xml`: improved side-chain torsional parameters of AMBER ff99SB for isoleucine, leucine, aspartate and asparagine, based on *ab initio* calculations and NMR data. In the literature referred to as AMBER ff99SB-ILDN. [REF](https://doi.org/10.1002/prot.22711)

- 2010 `amber99sbnmr.xml`: another (not as popular) refitting of the AMBER ff99SB backbone torsional parameters to NMR data. [REF](https://doi.org/10.1002/anie.201001898)

- 2010 `amber10.xml`: corresponds to AMBER ff10, which combines the AMBER ff99SB protein parameters with updated nucleic acid parameters. Its parameter file (`parm10.dat`) is also the starting point of AMBER ff14SB.

- 2015 `amber14-all.xml` or `amber14/protein.ff14SB.xml`: improved side-chain and backbone parameters starting from AMBER ff99SB, in the literature referred to as AMBER ff14SB. The file `amber14-all.xml` also includes parameters for DNA, RNA and lipids. [REF](https://doi.org/10.1021/acs.jctc.5b00255)

- 2020 `amber19-all.xml` or `amber19/protein.ff19SB.xml`: amino-acid-specific backbone parameters fitted to *ab initio* data, in the literature referred to as AMBER ff19SB. It is meant to be combined with the OPC water model. [REF](https://doi.org/10.1021/acs.jctc.9b00591)

One of the most widely used AMBER force fields is **AMBER ff14SB**, which is used throughout this course.
If for some reason, you need to use an older model, **AMBER ff99SB-ILDN** or **AMBER ff99SB** could be useful.
The remaining older ones are rarely used anymore.
The more recent **AMBER ff19SB** will not be used in this course.

Another popular family of biomolecular force fields are the CHARMM force fields. The [CHARMM36](https://doi.org/10.1021/ct300400x) force field was published in 2012 and its development followed similar steps as that of AMBER. In 1992 the CHARMM22 force field was released. In 2004, so-called [CMAP corrections](https://doi.org/10.1021/ja036959e) were released based on *ab initio* reference data, to improve the accuracy of the backbone conformations. In CHARMM36 this was taken one (big) step further, with more accurate *ab initio* data and follow-up refinements against NMR data.
OpenMM includes CHARMM36 as `charmm36.xml` and `charmm36/*.xml`.
The file `charmm36_2024.xml` contains a more recent release of the CHARMM parameters, including the [CHARMM36m](https://doi.org/10.1038/nmeth.4067) protein force field.
The CHARMM force fields will not be used in this course.


## 1. Gas phase (AMBER ff14SB)

All imports and initialization are put in the first cell.

In [ ]:
%matplotlib widget
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import openmm as mm
import pandas
from openmm import app, unit

We will run test simulations on a tiny "protein": alanine dipeptide.
The PDB file `alanine-dipeptide.pdb` for this notebook was taken from the [OpenMM test systems](https://github.com/openmm/openmm/tree/master/wrappers/python/tests/systems), where it is called `alanine-dipeptide-implicit.pdb`.

The simulation code below closely follows the water example from the first notebook ([01_first_steps/01_water.ipynb](../01_first_steps/01_water.ipynb)) with a few minor differences:

- The topology and the initial positions are now taken from a PDB file.

- A Langevin integrator (`LangevinMiddleIntegrator`) is used, instead of the Verlet integrator, with more MD steps.

- The AMBER ff14SB force field is used.

- X-H bonds are constrained in length, where X can be any atom. This allows us to take time steps of 2 femtoseconds.

- No initial velocities are assigned, so the atoms start at rest. The Langevin thermostat quickly heats the system to the target temperature.

This is an unrealistic simulation because the dipeptide is simulated in the gas phase (no solvation).
Without periodic boundary conditions, OpenMM computes all nonbonded interactions without a cutoff by default.

In [ ]:
pdb = app.PDBFile("alanine-dipeptide.pdb")
print(pdb.topology)
forcefield = app.ForceField("amber14-all.xml")
system = forcefield.createSystem(pdb.topology, constraints=app.HBonds)
integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin, 1 / unit.picosecond, 2 * unit.femtoseconds
)
simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()


**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Modify the PDB file to understand which pieces of information are essential to apply the force-field definition. Try making changes to:
>
> 1. the atom names, 
> 2. the name of a residue, 
> 3. the order of the atoms (within one residue or mixing residues) and 
> 4. the presence of atoms.
>
> Also note that the PDB file contains no bonds. These are somehow reconstructed when loading the PDB file. Try displacing one atom over a large distance. Does this affect the bond detection?

In the next cell, an MD simulation is carried out and the trajectory is written to a DCD file. This is a compact binary file format for trajectory data. Not only does this save disk space, loading DCD files is also much faster compared to PDB trajectory files.

In [ ]:
simulation.reporters = []
simulation.reporters.append(app.DCDReporter("traj1.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars1.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(100000)

In the next code cell, the potential energy as function of time is plotted. It reveals a very short equilibration phase. Because the energy was first minimized and the atoms start at rest, the potential energy starts low and increases quickly as the thermostat sets the atoms in motion.

Normally, the equilibration phase should be discarded prior to further analysis. For now, the equilibration phase is included in the analysis to explain and show this aspect.

In [ ]:
df1 = pandas.read_csv("scalars1.csv")
df1 = df1.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("ener1")
fig, ax = plt.subplots(num="ener1")
df1.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax);

The visualization below shows a few changes in conformation of the dipeptide.

In [ ]:
traj1 = mdtraj.load("traj1.dcd", top="alanine-dipeptide.pdb")
traj1.superpose(traj1, 0)
nglview.show_mdtraj(traj1)

The following cell defines a function to draw a [Ramachandran plot](https://en.wikipedia.org/wiki/Ramachandran_plot) with MatPlotLib. As opposed to a conventional Ramachandran plot, data points represent different conformations (time steps) of the $\phi$ and $\psi$ angles.

In [ ]:
def plot_ramachandran(traj, num):
    """Generate a basic Ramachandran plot for a given trajectory.

    Parameters
    ----------
    traj
        An MDTraj trajectory object.
    num
        A unique name for the Matplotlib figure.
    """
    phis = np.degrees(mdtraj.compute_phi(traj)[1].ravel())
    psis = np.degrees(mdtraj.compute_psi(traj)[1].ravel())
    plt.close(num)
    fig = plt.figure(num=num, figsize=(7, 4))
    gs = fig.add_gridspec(2, 3)
    # Ramachandran plot
    ax1 = fig.add_subplot(gs[:2, :2])
    ax1.plot(phis, psis, "k+")
    ax1.axvline(0)
    ax1.axhline(0)
    ax1.set_xlim(-180, 180)
    ax1.set_ylim(-180, 180)
    ax1.set_xticks(np.linspace(-180, 180, 5))
    ax1.set_yticks(np.linspace(-180, 180, 5))
    ax1.set_xlabel("Phi [deg]")
    ax1.set_ylabel("Psi [deg]")
    # Phi(t) plot
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.plot(np.arange(len(phis)), phis, "k+")
    ax2.axhline(0)
    ax2.set_ylim(-180, 180)
    ax2.set_yticks(np.linspace(-180, 180, 5))
    ax2.set_xlabel("Frame")
    ax2.set_ylabel("Phi [deg]")
    # Psi(t) plot
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(np.arange(len(psis)), psis, "k+")
    ax3.axhline(0)
    ax3.set_ylim(-180, 180)
    ax3.set_yticks(np.linspace(-180, 180, 5))
    ax3.set_xlabel("Frame")
    ax3.set_ylabel("Psi [deg]")


# Function call to make the plot
plot_ramachandran(traj1, "ram1")

## 2. Implicit solvent model (AMBER ff14SB with GBSA-OBC)

OpenMM supports various parameterizations of the generalized-Born Surface Area (GBSA) model, e.g. that by Onufriev, Bashford and Case (OBC), see https://doi.org/10.1002/prot.20033.
These implicit solvent models are taken from the [AMBER](https://ambermd.org/) program and can be combined with the AMBER and CHARMM force fields in OpenMM, by adding one of the following files to the force field definition:

- `implicit/hct.xml`
- `implicit/obc1.xml`
- `implicit/obc2.xml`
- `implicit/gbn.xml`
- `implicit/gbn2.xml`

See the [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#implicit-solvent) for more details.
Below, we use the OBC model (variant II) together with AMBER ff14SB.
Because the force field is the same as in the gas-phase simulation, differences between both simulations can be attributed to the solvent model.

(OpenMM also includes older files for implicit solvent, e.g. `amber99_obc.xml`, which are designed for the older AMBER force fields.
These will not be used in this course.)

In [ ]:
forcefield = app.ForceField("amber14-all.xml", "implicit/obc2.xml")
system = forcefield.createSystem(pdb.topology, constraints=app.HBonds)
integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin, 1 / unit.picosecond, 2 * unit.femtoseconds
)
simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()

simulation.reporters = []
simulation.reporters.append(app.DCDReporter("traj2.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars2.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(100000)

With the implicit solvent model, the computational cost increases by approximately 70% on a CPU, as can be seen from the elapsed times printed during both simulations.
This increase may vary with system size and hardware.

The potential energy as a function of time shows a short equilibration phase, comparable to the previous example:

In [ ]:
df2 = pandas.read_csv("scalars2.csv")
df2 = df2.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("ener2")
fig, ax = plt.subplots(num="ener2")
df2.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax);

We can now compare the structure of the backbone with and without implicit solvent. Also compare with figures obtained by other students.

In [ ]:
traj2 = mdtraj.load("traj2.dcd", top="alanine-dipeptide.pdb")
plot_ramachandran(traj2, "ram2")

## 3. Explicit solvent model (AMBER ff14SB with TIP3P-FB)

Any implicit solvent model will always be a serious approximation of explicit water molecules surrounding a solute. Before we can run such a simulation, we need to construct a topology and initial geometry of alanine dipeptide with a large number of water molecules. This can be done in several ways and for simplicity, we will use the built-in tools from OpenMM. For this example, we can again start from a PDB file.

This example is based on the following parts of the OpenMM documentation:

* https://docs.openmm.org/latest/userguide/application/03_model_building_editing.html
* https://docs.openmm.org/latest/api-python/generated/openmm.app.modeller.Modeller.html


One is in principle free to combine any biomolecule with any water force field, but not all combinations may have been carefully tested. Always check the original papers in which the force field was published to select matching pairs. To avoid confusion, the latest AMBER and CHARMM force fields in OpenMM are bundled with matching water force fields. Here are a few sensible combinations:

* `amber14-all.xml` can be paired with any water model in the `amber14` directory (TIP3P, TIP3P-FB, TIP4P-Ew, TIP4P-FB, SPC/E, OPC or OPC3), see https://github.com/openmm/openmm/tree/master/wrappers/python/openmm/app/data/amber14

* `amber19-all.xml` should preferably be combined with OPC (`amber19/opc.xml`), see https://github.com/openmm/openmm/tree/master/wrappers/python/openmm/app/data/amber19

* `charmm36.xml` should be combined with water force fields in the `charmm36` directory, see https://github.com/openmm/openmm/tree/master/wrappers/python/openmm/app/data/charmm36. Note that `charmm36/water.xml` is a slightly modified form of TIP3P.

* The parameters in `amber99sbildn.xml` [were tested](https://doi.org/10.1002/prot.22711) with TIP3P (`tip3p.xml`) and TIP4P-Ew (`tip4pew.xml`), see https://github.com/openmm/openmm/tree/master/wrappers/python/openmm/app/data.

The files from the GitHub repository should normally never be downloaded and links are only provided for reference. These files are included with any OpenMM installation and are already present on your machine.

In [ ]:
pdb = app.PDBFile("alanine-dipeptide.pdb")
modeller = app.Modeller(pdb.topology, pdb.positions)
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
# The model argument selects the geometry of the water molecules to be added.
# TIP3P-FB is a three-site model, just like TIP3P.
modeller.addSolvent(forcefield, model="tip3p", padding=1.2 * unit.nanometer)
print(modeller.topology)
# Write a PDB file to provide a topology of the solvated
# system to MDTraj below.
with open("init3.pdb", "w") as outfile:
    app.PDBFile.writeFile(modeller.topology, modeller.positions, outfile)

# The modeller builds a periodic box with the solute and solvent molecules.
# PME is the method to compute long-range electrostatic interactions in
# periodic systems.
system = forcefield.createSystem(modeller.topology, nonbondedMethod=app.PME, constraints=app.HBonds)
temperature = 300 * unit.kelvin
pressure = 1 * unit.bar
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, 2 * unit.femtoseconds)
system.addForce(mm.MonteCarloBarostat(pressure, temperature))
simulation = app.Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy()
simulation.reporters.append(app.DCDReporter("traj3.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars3.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(100000)

This computation was considerably slower: on a CPU, it is roughly a factor of 8 more expensive than the gas-phase simulation.

Before going further, we plot the potential energy as function of time, to estimate the length of the equilibration phase. There was barely any equilibration in the previous two runs, mainly because the few degrees of freedom were all relatively fast motions. More complex structures tend to exhibit also slower motions. For example, in our current run with explicit water molecules, the solvent needs more time to equilibrate.

In [ ]:
df3 = pandas.read_csv("scalars3.csv")
df3 = df3.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("ener3")
fig, ax = plt.subplots(num="ener3")
df3.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax);

The figure above shows that approximately (the first) 15 picoseconds are required for the equilibration. Results for these steps need to be removed before performing any analysis. A single MD step takes 2 femtoseconds and only every 100 steps, a frame is written to the DCD file, which means that the first 75 frames from the trajectory should be removed. For the visualization, we still look at all steps.

In [ ]:
traj3 = mdtraj.load("traj3.dcd", top="init3.pdb")
view = nglview.show_mdtraj(traj3)
view.clear_representations()
view.add_licorice()
view.add_unitcell()
view

A few remarks on the visualization:

- The absolute position of the dipeptide relative to the box is not crucial. Due to the periodic boundary conditions, all molecules interact with an infinitely large environment, of which the visualization only shows a small fragment.

- The dipeptide slowly diffuses through the liquid, which is the expected behavior. Water molecules make larger jumps because they are smaller and lighter.

In the next cell, the equilibration phase is discarded and the water molecules are removed, before making the Ramachandran plot.

In [ ]:
traj3.restrict_atoms(traj3.topology.select("protein"))
plot_ramachandran(traj3[75:], "ram3")

The Ramachandran plot is strongly influenced by the choice of solvent model. Already for a simple dipeptide, the limitations of an implicit solvent model are clear. The reason is that the amide groups participate in the hydrogen-bonding network of the solvent. These are rather specific and directional interactions, which are absent in implicit solvent models.

Another difference with the first two runs is the slower change of the backbone angles. There are fewer changes between conformations and switching also takes longer. This is due to the friction with and the inertia of the surrounding water molecules. Hence, a single time step with explicit solvent is more costly and one has to perform longer simulations because water slows down conformational changes. For this reason, implicit solvent models are still popular, despite the fact that they are very approximate. Note that the explicit-solvent simulations in this notebook are not long enough to obtain converged statistics.